# Feature Engineering
- Extrair, dos dados brutos, os melhores recursos (features) para o nosso modelo para aumentar a acurácia

In [2]:
import pandas as pd

In [21]:
clean_base = pd.read_excel("../../data/ChavesClientes.xlsx")
clean_base.head()

,ID,ChaveSituacao,ClassRisco,CatCliente,Pagamento
0,1,32FC,Ccinza,Basic-Alpha,1
1,2,25MV,AAmarelo,Black,1
2,3,27MV,B-Amarelo,Basic-Beta,1
3,4,26FD,BAmarelo,Black,0
4,5,26FD,C-Amarelo,Black,0


In [22]:
clean_base = pd.read_excel("../../data/ChavesClientesLimpo.xlsx")
clean_base.head()

,ChaveSituacao,ClassRisco,CatCliente,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco
0,32FC,Ccinza,Basic-Alpha,1,32,F,C,Basic,Alpha,C
1,25MV,AAmarelo,Black,1,25,M,V,Black,Comum,A
2,27MV,B-Amarelo,Basic-Beta,1,27,M,V,Basic,Beta,B-
3,26FD,BPreto,Black,0,26,F,D,Black,Comum,B
4,26FD,C-Amarelo,Black,0,26,F,D,Black,Comum,C-


**Podemos excluir as colunas que não vamos usar**

In [23]:
clean_base = clean_base.drop(['ChaveSituacao', 'ClassRisco', 'CatCliente'], axis=1)

**E verificar as informações da base**

In [24]:
clean_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Pagamento    20 non-null     int64
 1   Idade        20 non-null     int64
 2   Genero       20 non-null     str  
 3   EstadoCivil  20 non-null     str  
 4   Categoria    20 non-null     str  
 5   CatVIP       20 non-null     str  
 6   Risco        20 non-null     str  
dtypes: int64(2), str(5)
memory usage: 1.2 KB


### Ao tentar colocar esses dados em um modelo como o de Regressão Linear vamos ter o seguinte erro

In [25]:
# Selecionando os valores de X e y
X = clean_base[['Idade', 'Genero', 'EstadoCivil', 'Categoria', 'CatVIP', 'Risco']]
y = clean_base.Pagamento

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X, y)

reg.score(X,y)

ValueError: could not convert string to float: 'F'

### Por isso precisamos conseguir tratar os dados antes de inserir no modelo

In [26]:
clean_base.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco
0,1,32,F,C,Basic,Alpha,C
1,1,25,M,V,Black,Comum,A


**Com o One Hot Encoding podemos tratar valores que não tem relação de ordem entre eles**
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

In [27]:
# Importando e utilizando o OneHotEncoder para as colunas 'Genero' e 'EstadoCivil'
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
ohe_transform = ohe.fit_transform(clean_base[['Genero', 'EstadoCivil']])

In [30]:
# Nome das features
ohe.get_feature_names_out()

array(['Genero_F', 'Genero_M', 'EstadoCivil_C', 'EstadoCivil_D',
       'EstadoCivil_S', 'EstadoCivil_V'], dtype=object)

In [31]:
# Array de valores
ohe_transform.toarray()

array([[1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0.],
       [1., 0., 0., 1., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [1., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 1.],
       [1., 0., 1., 0., 0., 0.],
       [1., 0., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [0., 1., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 1.]])

In [32]:
# Transformando esses dados em um DataFrame
df_ohe = pd.DataFrame(ohe_transform.toarray())
df_ohe.columns = ohe.get_feature_names_out()
df_ohe.head()

,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V
0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,1.0,0.0,0.0


In [33]:
# Para finalizar, podemos concatenar as duas bases
clean_base = pd.concat([clean_base, df_ohe], axis=1)

In [34]:
clean_base.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0


**Já se os valores tiverem uma relação de ordem, podemos usar o Ordinal Encoding**
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder

In [35]:
# Entendendo a relação entre a coluna "Categoria"
clean_base.Categoria.value_counts()

Categoria
Black       7
Platinum    7
Basic       6
Name: count, dtype: int64

In [36]:
# Importando e utilizando o OrdinalEncoder para a coluna 'Categoria'
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder()
oe_transform = oe.fit_transform(clean_base.Categoria.values.reshape(-1, 1))

In [88]:
oe_transform

array([[0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [2.],
       [2.],
       [0.],
       [1.],
       [2.],
       [0.],
       [0.],
       [0.],
       [2.],
       [1.],
       [2.],
       [1.],
       [1.],
       [2.],
       [2.]])

In [37]:
# E podemos adicionar essa coluna
clean_base['NrCategoria'] = oe_transform

In [38]:
# Visualizando a base
clean_base.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0,1.0


In [39]:
# Fazendo o mesmo para a coluna risco
oe = OrdinalEncoder(categories=[['C-','C','C+','B-','B','B+','A-','A','A+']])
oe_transform_risco = oe.fit_transform(clean_base.Risco.values.reshape(-1, 1))
oe_transform_risco

array([[1.],
       [7.],
       [3.],
       [4.],
       [0.],
       [0.],
       [6.],
       [0.],
       [6.],
       [2.],
       [7.],
       [0.],
       [4.],
       [6.],
       [1.],
       [3.],
       [6.],
       [0.],
       [1.],
       [7.]])

In [40]:
clean_base['NrRisco'] = oe_transform_risco

In [41]:
clean_base.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0


**Por fim, podemos criar funções para transformar colunas como transformar a CatVIP para verificar apenas se o cliente é VIP ou não**

In [42]:
# Criando uma função para verificar se o cliente é VIP
def define_vip(valor):
    if valor == 'Alpha' or valor == 'Beta':
        return 1
    else:
        return 0

In [43]:
# Aplicando essa função na coluna 'CatVIP'
clean_base['NrVIP'] = clean_base.CatVIP.apply(define_vip)

In [44]:
clean_base.head(2)

,Pagamento,Idade,Genero,EstadoCivil,Categoria,CatVIP,Risco,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco,NrVIP
0,1,32,F,C,Basic,Alpha,C,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
1,1,25,M,V,Black,Comum,A,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0,0


**Limpando novamente as colunas desnecessárias**

In [45]:
# Retirando novamente as colunas desnecessárias
clean_base = clean_base.drop(['Genero', 'EstadoCivil', 'Categoria', 'CatVIP', 'Risco'], axis=1)
clean_base.head()

,Pagamento,Idade,Genero_F,Genero_M,EstadoCivil_C,EstadoCivil_D,EstadoCivil_S,EstadoCivil_V,NrCategoria,NrRisco,NrVIP
0,1,32,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
1,1,25,0.0,1.0,0.0,0.0,0.0,1.0,1.0,7.0,0
2,1,27,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,1
3,0,26,1.0,0.0,0.0,1.0,0.0,0.0,1.0,4.0,0
4,0,26,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0


### Usando novamente em um modelo de Regressão Linear

In [47]:
# Selecionando os valores de X e y
X = clean_base.drop('Pagamento', axis=1)
y = clean_base.Pagamento

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X, y)

reg.score(X,y)

0.6197521275369209

In [48]:
# Selecionando os valores de X e y
X = clean_base.Idade.values.reshape(-1, 1)
y = clean_base.Pagamento

from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X, y)

reg.score(X,y)

0.03832882093751644